# Hotspot & cluster analysis (ESDA)

End-to-end walkthrough: load areal units + variable, build and compare
spatial weights, test global autocorrelation (Moran's I), then map local
clusters (LISA) and hot/cold spots (Getis-Ord Gi*).

All analysis choices come from `config/aoi.yaml`. Interpretation caveats are
in the final cell and the README.

## 0. Hypothesis

County corn yield in Iowa (2023) is **positively spatially autocorrelated**
(soil/climate gradients), so we expect a significant positive global Moran's
I and coherent High-High / Low-Low local clusters.

In [ ]:
import geopandas as gpd
import yaml

from hotspots import esda
from hotspots.weights import compare_weights, queen_contiguity, knn, distance_band

with open("../config/aoi.yaml") as fh:
    cfg = yaml.safe_load(fh)
value_col = cfg["variable"]["value_column"]
inf = cfg["inference"]
cfg["aoi"]["name"]

In [ ]:
# Built by `python data/download.py` (or `make download`).
gdf = gpd.read_file(f"../data/raw/{cfg['aoi']['name']}.gpkg").to_crs(cfg["aoi"]["crs"])
values = gdf[value_col].to_numpy(float)
gdf.plot(column=value_col, legend=True, scheme="quantiles", k=5, cmap="viridis")

## 1. Build and compare weights

The choice of W is a modelling decision; compare neighbour structure before
committing. Islands raise by default.

In [ ]:
w_queen = queen_contiguity(gdf, on_islands="warn")
w_knn = knn(gdf, k=cfg["weights"]["knn"]["k"])
w_dist = distance_band(gdf, on_islands="warn")
for d in compare_weights({"queen": w_queen, "knn": w_knn, "distance": w_dist}):
    print(d.as_dict())

w = w_queen  # primary

## 2. Global Moran's I (permutation inference)

In [ ]:
gm = esda.global_moran(values, w, permutations=inf["permutations"], seed=inf["seed"])
print(gm)
esda.moran_scatterplot(values, w)

## 3. Local clusters: LISA and Getis-Ord Gi*

In [ ]:
lisa = esda.local_moran(values, w, permutations=inf["permutations"],
                        significance=inf["significance"], seed=inf["seed"])
gi = esda.getis_ord_gi_star(values, w, permutations=inf["permutations"],
                            significance=inf["significance"], seed=inf["seed"])
gdf["lisa"], gdf["gi"] = lisa.labels, gi.labels
gdf.plot(column="lisa", categorical=True, legend=True, cmap="Set1")

## 4. Interpretation and caveats

- Clusters are **descriptive, not causal**.
- Results are **conditional on W** (we compared three) and on **stationarity**.
- LISA/Gi* are **multiple-tested**; treat pseudo p as flags and correct (FDR).
- Subject to **MAUP** (county aggregation).

Optional next step: `hotspots.gwr.fit_gwr` to probe spatially varying
relationships once covariates are added to the config.